# global 指標の epoch 推移 — iterative 4 条件と baseline

変調範囲（`fc` / `stage4+fc`）× GroupDRO step size（1e-3 / 1e-2）の iterative 4 run と、
通常の ResNet（ERM、30 epoch）について、**validation の global AUROC・balanced accuracy・loss** が
epoch ごとにどう動いたかを並べる。

問いは 2 つある。

1. iterative は warmup → stage01 → stage02 の各段で、global 性能をどれだけ失っているか。
2. 選ばれた checkpoint（stage02 の best val AUROC）は、推移のどこに位置しているか。

全 run が seed 42 の 1 本なので、ここで読めるのは推移の形であり、条件間の差の有意性ではない。
群ごとの推移（cohort の `q` と `hidden_*`）は別の notebook で扱う。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.style
import pandas as pd
import rootutils
import torch

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)

from analysis.common.paths import STYLE_SHEET, run_dir  # noqa: E402
from analysis.common.run_artifacts import read_wandb_history, selected_checkpoint  # noqa: E402
from analysis.iterative_probe._shared import collect_epoch_rows  # noqa: E402

matplotlib.style.use(STYLE_SHEET)

## 対象 run と定数

iterative の 4 run は条件順（fc 1e-3 → fc 1e-2 → stage4+fc 1e-3 → stage4+fc 1e-2）に並べる。
表と図の並びはこの順になる。baseline は `hypernet_e2e` で回した ERM で、split の sha256・optimizer・
batch size・class weight・seed が iterative 側と一致する（[runs.md](runs.md)）。

iterative は 12 epoch（warmup 2 + stage01 5 + stage02 5）、baseline は 30 epoch 回している。
**同じ epoch 予算で並べるときは baseline の epoch 0〜11 を使い**、baseline の 30 epoch 全体は
到達点の参照として別に表へ出す。

In [ ]:
STUDY = "iterative_probe"
ITERATIVE_RUN_IDS = [
    "20260924T112035Z-spatial-lora-iterative-chexpert-fc-s42-b81a",
    "20260924T112045Z-spatial-lora-iterative-chexpert-fc-s42-4602",
    "20260924T112034Z-spatial-lora-iterative-chexpert-stage4-fc-s42-1caa",
    "20260924T112043Z-spatial-lora-iterative-chexpert-stage4-fc-s42-c755",
]
BASELINE_RUN_ID = "20260921T103036Z-resnet-chexpert-s42-5538"
BASELINE_CONDITION = "ResNet ERM"

METRICS = {
    "val/auroc": "Validation AUROC",
    "val/bacc": "Validation balanced accuracy",
    "val/loss": "Validation loss (unweighted CE)",
}
MODULATIONS = ["fc", "stage4+fc"]
STEP_COLOR = {0.001: "#0173B2", 0.01: "#DE8F05"}
BASELINE_STYLE = {"color": "#000000", "marker": "s", "linestyle": ":", "markersize": 3.5}
STAGE_NAMES = ["warmup", "stage01", "stage02"]

RESULTS = ROOT / "analysis" / STUDY / "results"
FIGURES = ROOT / "analysis" / STUDY / "figures"

## iterative の epoch 推移

各 stage の `stages/<stage>/metrics/metrics.csv` を読み、warmup から通した epoch 番号（`run_epoch`、0 始まり）を振る。
変調範囲と step size は各 run の `config.yaml` から読む（`_shared.collect_epoch_rows`）。
stage の境目で cohort が引き直され、GroupDRO の `q` も初期化される。

In [ ]:
iterative_rows = [row for run_id in ITERATIVE_RUN_IDS for row in collect_epoch_rows(run_dir(STUDY, run_id))]
iterative = pd.DataFrame(iterative_rows)[
    ["run_id", "condition", "modulation", "step_size", "stage", "run_epoch", "epoch", *METRICS]
]
iterative.pivot_table(index="run_epoch", columns="condition", values="val/auroc", sort=False).round(4)

## baseline の epoch 推移

baseline は CSVLogger を付けずに回したため、epoch 推移は W&B の transaction log（`wandb/run-*/run-*.wandb`）
にしか残っていない。`read_wandb_history` で epoch ごとの 1 行に束ね直す。

In [ ]:
baseline_history = pd.DataFrame(read_wandb_history(run_dir(STUDY, BASELINE_RUN_ID)))
# baseline は変調範囲と step size を持たないので、その 2 列は作らない（結合後は欠損になる）。
baseline_columns = [column for column in iterative.columns if column not in ("modulation", "step_size")]
baseline = baseline_history.assign(
    run_id=BASELINE_RUN_ID,
    condition=BASELINE_CONDITION,
    stage="erm",
    run_epoch=baseline_history["epoch"].astype(int),
)[baseline_columns]
len(baseline), baseline["run_epoch"].min(), baseline["run_epoch"].max()

### loss を並べてよいか

`val/loss` は両 project とも `validation_step` で `F.cross_entropy(logits, target)` を取っており、
**class weight も GroupDRO の重みも掛けない素の cross-entropy** である。したがって val loss は
同じ定義で並べられる。

一方 `train/loss` は並べない。baseline は class weight 付き cross-entropy、iterative の stage は
GroupDRO の目的関数（cohort ごとの損失を `q` で重み付けしたもの）であり、値の意味が違う。

学習は inverse class weight で行っているので、重みの無い val loss は陽性の少ない不均衡をそのまま
反映し、閾値付近の確率の置き方で大きく振れる。loss は**水準より推移の形**として読む。

In [ ]:
epoch_metrics = pd.concat([iterative, baseline], ignore_index=True)
RESULTS.mkdir(parents=True, exist_ok=True)
epoch_metrics.to_csv(RESULTS / "global_epoch_metrics.csv", index=False)
RESULTS / "global_epoch_metrics.csv"

## 選択 checkpoint の位置

図に選択 checkpoint を重ねるため、各 run の checkpoint が何 epoch 目かを求める。
file 名の番号ではなく **checkpoint 自身が持つ `epoch`** を読み、stage 内の epoch を通し epoch に直す。

`run.json` の `selected_checkpoint.score` は best ではなく stage 最終 epoch の値になっている
（[runs.md](runs.md) の「選択 checkpoint」）。ここでは score を読まず、val AUROC は `metrics.csv` 側から引く。

In [ ]:
def checkpoint_epoch(path: Path) -> int:
    # checkpoint は 1 本 270MB あるので、epoch だけを取り出したら捨てる。
    return int(torch.load(path, map_location="cpu", weights_only=False)["epoch"])


def stage_offsets(frame: pd.DataFrame) -> dict[str, int]:
    return frame.groupby("stage", sort=False)["run_epoch"].min().to_dict()


selected_rows = []
for run_id, part in epoch_metrics.groupby("run_id", sort=False):
    path = selected_checkpoint(run_dir(STUDY, run_id))
    # iterative は stages/<stage>/checkpoints/、baseline は checkpoints/ に置く。
    stage = path.parent.parent.name if path.parent.parent.parent.name == "stages" else "erm"
    run_epoch = stage_offsets(part)[stage] + checkpoint_epoch(path)
    at_epoch = part.set_index("run_epoch").loc[run_epoch]
    selected_rows.append(
        {
            "run_id": run_id,
            "condition": at_epoch["condition"],
            "checkpoint": f"{stage}/{path.name}",
            "run_epoch": run_epoch,
            **{metric: at_epoch[metric] for metric in METRICS},
        }
    )

selected = pd.DataFrame(selected_rows)
selected

## 推移の図

変調範囲ごとに行を分け、線の色は step size だけを表す（`plots.py` と同じ規約）。4 条件を 1 枚に
重ねると線が交差して読めないためである。baseline は条件の 1 つではなく参照線として、黒の点線で
両方の行に同じものを描く。縦の破線は stage の境目、★ は選択 checkpoint を表す。

横軸は iterative と同じ予算（epoch 0〜11）に切る。baseline の 12 epoch 以降は次の表で扱う。

In [ ]:
BUDGET = int(iterative["run_epoch"].max())
first_run = iterative[iterative["run_id"] == ITERATIVE_RUN_IDS[0]]
boundaries = [epoch - 0.5 for epoch in stage_offsets(first_run).values()][1:]
baseline_in_budget = baseline[baseline["run_epoch"] <= BUDGET]

figure, axes = plt.subplots(len(MODULATIONS), len(METRICS), figsize=(14, 7), sharex=True, sharey="col")
for row, modulation in enumerate(MODULATIONS):
    for column, (metric, title) in enumerate(METRICS.items()):
        axis = axes[row, column]
        axis.plot(
            baseline_in_budget["run_epoch"], baseline_in_budget[metric], label=BASELINE_CONDITION, **BASELINE_STYLE
        )
        for run_id, part in iterative[iterative["modulation"] == modulation].groupby("run_id", sort=False):
            step_size = part["step_size"].iloc[0]
            axis.plot(
                part["run_epoch"],
                part[metric],
                marker="o",
                markersize=3.5,
                linewidth=1.6,
                color=STEP_COLOR[step_size],
                label=f"step size {step_size:g}",
            )
            chosen = selected[selected["run_id"] == run_id].iloc[0]
            axis.plot(
                chosen["run_epoch"],
                chosen[metric],
                marker="*",
                markersize=13,
                color=STEP_COLOR[step_size],
                markeredgecolor="white",
                linestyle="none",
            )
        for boundary in boundaries:
            axis.axvline(boundary, color="#9a9a96", linestyle="--", linewidth=0.8)
        if row == 0:
            axis.set_title(title)
        if column == 0:
            axis.set_ylabel(modulation)
        if row == len(MODULATIONS) - 1:
            axis.set_xlabel("epoch (warmup → stage01 → stage02)")
        axis.set_xticks(range(BUDGET + 1))

handles, labels = axes[0, 0].get_legend_handles_labels()
figure.legend(handles, labels, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.04))
figure.savefig(FIGURES / "global_training_curves_val.png", dpi=180, bbox_inches="tight")
plt.show()

## 到達点の表

同じ推移を、読み取りに使う点だけ表にする。

- `warmup_end`: warmup 最終 epoch（epoch 1）。GroupDRO が入る前の到達点
- `budget_end`: 12 epoch 目（epoch 11）。iterative の最終 epoch と同じ予算
- `best`: 記録のある全 epoch の中で val AUROC が最大の epoch
- `selected`: 分析に使う checkpoint

baseline は 30 epoch 全体を持つので、`best` は 12 epoch 以降も含む。

In [ ]:
def snapshot(part: pd.DataFrame, name: str, run_epoch: int) -> dict[str, object]:
    at_epoch = part.set_index("run_epoch").loc[run_epoch]
    return {
        "condition": at_epoch["condition"],
        "point": name,
        "run_epoch": run_epoch,
        **{metric: at_epoch[metric] for metric in METRICS},
    }


snapshot_rows = []
for run_id, part in epoch_metrics.groupby("run_id", sort=False):
    best_epoch = int(part.loc[part["val/auroc"].idxmax(), "run_epoch"])
    chosen_epoch = int(selected.loc[selected["run_id"] == run_id, "run_epoch"].iloc[0])
    points = [("warmup_end", 1), ("budget_end", BUDGET), ("best", best_epoch), ("selected", chosen_epoch)]
    for name, run_epoch in points:
        snapshot_rows.append(snapshot(part, name, run_epoch))

snapshots = pd.DataFrame(snapshot_rows)
snapshots.to_csv(RESULTS / "global_epoch_snapshots.csv", index=False)
snapshots.pivot_table(index="condition", columns="point", values="val/auroc", sort=False)[
    ["warmup_end", "budget_end", "best", "selected"]
].round(4)

balanced accuracy と loss も同じ点で見る。

In [ ]:
snapshots.pivot_table(index="condition", columns="point", values=["val/bacc", "val/loss"], sort=False).round(4)

### stage ごとの振れ幅

stage の中で val AUROC がどれだけ上下したかを、stage ごとの max − min で見る。stage4+fc の揺れが
fc より大きいかを確かめるための表である。baseline は stage を持たないので、比較のため
epoch 2〜6 と 7〜11（iterative の stage01 / stage02 と同じ epoch）で区切る。

In [ ]:
baseline_as_stages = baseline_in_budget.assign(
    stage=pd.cut(baseline_in_budget["run_epoch"], bins=[-1, 1, 6, BUDGET], labels=STAGE_NAMES).astype(str)
)
stage_range = (
    pd.concat([iterative, baseline_as_stages])
    .groupby(["condition", "stage"], sort=False)["val/auroc"]
    .agg(lambda values: values.max() - values.min())
    .unstack("stage")[STAGE_NAMES]
)
stage_range.round(4)

## まとめ

数値の正本は `results/global_epoch_metrics.csv` と `results/global_epoch_snapshots.csv`。
読み取りの記録は [reports/global_training_curves.md](reports/global_training_curves.md) に残す。

**観察**

- **warmup の到達点は baseline と並ぶ。** epoch 1 の val AUROC は fc 0.8580、stage4+fc 0.8545、
  baseline 0.8559。同じ変調範囲の 2 run は warmup が一致しており、step size の違いは stage01 から効く。
- **GroupDRO の stage に入ると global AUROC は warmup の水準に戻らない。** 4 run とも全 12 epoch の
  best は warmup 最終 epoch で、12 epoch 目の val AUROC は 0.830〜0.851。同じ予算で baseline は 0.8605、
  30 epoch の best は 0.8675（epoch 9）まで上がる。
- **選択 checkpoint（stage02 の best）は 0.8482〜0.8539。** warmup 最終 epoch より 0.004〜0.007 低い。
- **stage4+fc は stage02 で大きく振れる。** stage 内の val AUROC の max − min は stage4+fc が 0.018 / 0.020、
  fc は 0.011 / 0.019。同じ epoch の区間で baseline は 0.009。stage4+fc の 2 本は stage02 の頭（epoch 7）で
  上がり、epoch 8 で 0.84 前後まで落ちる形が揃っている。
- **balanced accuracy は stage01 以降、baseline を下回る。** 4 run × 10 epoch の 40 点のうち、上回るのは 1 点だけ。 fc / 1e-2 は stage01 の
  epoch 4 で 0.747 まで落ち、同じ epoch で val loss が 0.72 に跳ねる。
- **val loss は baseline 自身も epoch 0〜11 で 0.36〜0.59 と振れる。** 水準の差として読めるほど安定していない。

**読めないこと**

- seed が 1 本なので、条件間の差（選択 checkpoint で最大 0.006）は読まない。baseline の epoch 間の
  振れ（stage01 相当の区間で 0.014）が同程度にある。
- stage の境目では optimizer（AdamW）の状態も初期化される（引き継ぐのは net の重みだけ）。
  stage 頭の変化が cohort の引き直しによるのか、optimizer の初期化によるのかは、この推移からは分けられない。